# 4. Provjera importa podataka

Ovaj notebook verificira kvalitetu uvezenih podataka:
1. Usporedba broja redaka (CSV vs SQL)
2. Provjera duplikata
3. NULL analiza ključnih stupaca
4. Konzistentnost mapiranja
5. Profiliranje distribucija

In [1]:
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

DB_USER = os.getenv('DB_USER', 'root')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_NAME = os.getenv('DB_NAME', 'fipu_srp_projekt')

if not DB_PASSWORD:
    raise ValueError("DB_PASSWORD nije postavljen! Kreiraj .env datoteku.")

engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}")
print(f"Spojeno na bazu: {DB_NAME} ({DB_HOST})")

CSV_FILE_PATH = 'Support_tickets_PROCESSED.csv'

Spojeno na bazu: fipu_srp_projekt (localhost)


## 4.1 Usporedba broja redaka

In [2]:
df_csv = pd.read_csv(CSV_FILE_PATH)
csv_count = len(df_csv)

with engine.connect() as conn:
    db_count = conn.execute(text("SELECT COUNT(*) FROM support_tickets")).scalar()

print(f"CSV:  {csv_count} redaka")
print(f"SQL:  {db_count} redaka")
print(f"Match: {'DA' if csv_count == db_count else 'NE!'}")

CSV:  53353 redaka
SQL:  53353 redaka
Match: DA


## 4.2 Provjera duplikata

In [3]:
with engine.connect() as conn:
    dups = conn.execute(text(
        "SELECT id, COUNT(*) c FROM support_tickets GROUP BY id HAVING c > 1"
    )).fetchall()

if not dups:
    print("Nema duplih ID-ova.")
else:
    print(f"Pronađeno {len(dups)} duplikata!")

Nema duplih ID-ova.


## 4.3 NULL analiza ključnih stupaca

In [4]:
df = pd.read_sql("SELECT * FROM support_tickets", engine)
key_cols = ['id', 'issue_proj', 'issue_reporter', 'issue_assignee', 'issue_created']
null_counts = df[key_cols].isnull().sum()
print("NULL vrijednosti po ključnim stupcima:")
print(null_counts)

NULL vrijednosti po ključnim stupcima:
id                    0
issue_proj            0
issue_reporter        0
issue_assignee    24771
issue_created         0
dtype: int64


## 4.4 Konzistentnost mapiranja

In [5]:
with engine.connect() as conn:
    result = conn.execute(text(
        "SELECT issue_proj, COUNT(DISTINCT issue_proj) as cnt "
        "FROM support_tickets GROUP BY issue_proj HAVING cnt > 1"
    )).fetchall()

if not result:
    print("Mapiranje projekata je konzistentno (1 ID = 1 ime).")
else:
    print(f"Pronađena nekonzistentnost: {result}")

Mapiranje projekata je konzistentno (1 ID = 1 ime).


## 4.5 Profiliranje distribucija

In [6]:
df_profile = pd.read_sql(
    "SELECT issue_type, issue_priority, issue_status FROM support_tickets", engine
)
print("Distribucija prioriteta:")
print(df_profile['issue_priority'].value_counts())
print(f"\nDistribucija tipova:")
print(df_profile['issue_type'].value_counts())

Distribucija prioriteta:
issue_priority
unknown    27156
Medium     19901
High        3595
Highest     1636
Blocker      525
Low          469
Lowest        71
Name: count, dtype: int64

Distribucija tipova:
issue_type
Ticket            36172
Service            4343
Subtask            3745
Story              3619
HD Service         1340
Task               1240
Vacation            686
Project             680
Sub-task            431
Epic                325
Deployment          270
Retrospective       198
Sprint Summary      169
Assistance           88
Bug                  47
Name: count, dtype: int64


## 4.6 Uzorak podataka

In [7]:
sample = pd.read_sql("SELECT * FROM support_tickets LIMIT 5", engine)
print(sample.to_string(index=False))

   id             started               ended  issue_num    issue_proj issue_reporter issue_assignee  issue_contr_count issue_type issue_priority       issue_created issue_resolution_date issue_resolution issue_status  issue_comments_count    last_change_date  wfe_in_review  wfe_deployment wf_resolved  wfe_resolved  wf_open  wfe_open  wfe_monitoring  wfe_done  wfe_pending_customer_approval  wfe_rejected  wfe_testing_monitoring  wf_in_progress  wfe_in_progress  wfe_reopened  wfe_to_do  wfe_validation  wfe_resolved_under_monitoring  wfe_closed wf_waiting  wfe_waiting  wfe_cancelled  wfe_under_review  wfe_approved  wfe_pending_deployment  wf_total_time  processing_steps
11887 2016-01-06 08:23:43 2016-01-06 08:56:55        186 Project Atlas Petra Pavlović           None                  1     Ticket         Medium 2016-01-06 08:23:43   2016-01-06 08:56:55             Done         done                     1 2016-04-02 12:20:21              0               0        None             0     29.